# Comprehensive Performance Analysis: RQ1 & RQ2

This notebook provides a complete analysis across all experimental dimensions:

## Dimensions Analyzed
- **Tasks**: Code Generation, Vulnerability Detection, Log Analysis
- **Designs**: SA (Single-Agent), DA (Dual-Agent), MA (Multi-Agent)
- **Prompting**: Zero-shot vs Few-shot
- **Modes**: Instruct vs Thinking (reasoning)
- **Model Sizes**: 4B, 8B, 30B, 49B parameters

## Models
- Qwen3-4B-Instruct / Qwen3-4B-Thinking
- Nemotron-Nano-8B (Instruct / Thinking)
- Qwen3-30B-A3B-Instruct / Qwen3-30B-A3B-Thinking
- Nemotron-Super-49B (Instruct / Thinking)

**Date**: January 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# Base paths
from pathlib import Path
BASE_DIR = Path('..').resolve()
RESULTS_DIR = BASE_DIR / 'results'
ANALYSIS_DIR = RESULTS_DIR / 'analysis_comprehensive'
ANALYSIS_DIR.mkdir(exist_ok=True)

print(f"Results directory: {RESULTS_DIR}")
print(f"Analysis output: {ANALYSIS_DIR}")

## 1. Load and Prepare Data

In [ ]:
# Load consolidated performance data
perf_df = pd.read_csv(RESULTS_DIR / 'consolidated_performance.csv')

print(f"Total experiments: {len(perf_df)}")
print(f"\nBreakdown by task:")
print(perf_df['task'].value_counts())
print(f"\nBreakdown by design:")
print(perf_df['design'].value_counts())

In [ ]:
# Create unified performance metric column
# - Code generation: pass_at_1
# - Vulnerability detection: f1_score
# - Log analysis: f1_score

def get_primary_metric(row):
    if row['task'] == 'code_generation':
        return row['pass_at_1']
    else:
        return row['f1_score']

perf_df['primary_metric'] = perf_df.apply(get_primary_metric, axis=1)
perf_df['primary_metric_pct'] = perf_df['primary_metric'] * 100
perf_df['accuracy_pct'] = perf_df['accuracy'] * 100
perf_df['f1_pct'] = perf_df['f1_score'] * 100

# Create short labels
def make_short_label(row):
    params = int(row['parameters_b'])
    mode = 'T' if row['mode'] == 'thinking' else 'I'
    prompt = 'F' if row['prompting'] == 'few-shot' else 'Z'
    return f"{params}B-{mode}-{prompt}"

perf_df['short_label'] = perf_df.apply(make_short_label, axis=1)

# Display sample
perf_df[['short_label', 'task', 'design', 'primary_metric_pct']].head(10)

## 2. Color and Style Configuration

In [ ]:
# Color palettes
MODEL_COLORS = {
    4: '#3498db',    # Blue for 4B
    8: '#00bcd4',    # Cyan for 8B
    30: '#9b59b6',   # Purple for 30B
    49: '#e91e63',   # Magenta for 49B
}

DESIGN_COLORS = {
    'SA': '#2ecc71',  # Green
    'DA': '#3498db',  # Blue
    'MA': '#e74c3c',  # Red
}

TASK_COLORS = {
    'code_generation': '#3498db',
    'vulnerability_detection': '#e74c3c',
    'log_analysis': '#f39c12',
}

PROMPTING_COLORS = {
    'zero-shot': '#95a5a6',
    'few-shot': '#2c3e50',
}

MODE_HATCHES = {
    'instruct': '',
    'thinking': '//',
}

TASK_LABELS = {
    'code_generation': 'Code Generation',
    'vulnerability_detection': 'Vulnerability Detection',
    'log_analysis': 'Log Analysis',
}

METRIC_LABELS = {
    'code_generation': 'Pass@1 (%)',
    'vulnerability_detection': 'F1 Score (%)',
    'log_analysis': 'F1 Score (%)',
}

print("Color configuration loaded.")

## 3. Performance by Task (Overview)

In [ ]:
# Summary statistics by task
task_summary = perf_df.groupby('task').agg({
    'primary_metric_pct': ['mean', 'std', 'min', 'max'],
    'accuracy_pct': ['mean', 'std', 'min', 'max']
}).round(2)

print("="*80)
print("PERFORMANCE SUMMARY BY TASK")
print("="*80)
print(task_summary)

# Box plot by task
fig, ax = plt.subplots(figsize=(10, 6))

tasks_order = ['code_generation', 'vulnerability_detection', 'log_analysis']
task_data = [perf_df[perf_df['task'] == t]['primary_metric_pct'].dropna() for t in tasks_order]

bp = ax.boxplot(task_data, patch_artist=True, labels=[TASK_LABELS[t] for t in tasks_order])

# Color boxes
for patch, task in zip(bp['boxes'], tasks_order):
    patch.set_facecolor(TASK_COLORS[task])
    patch.set_alpha(0.7)

ax.set_ylabel('Performance (%)', fontsize=12)
ax.set_title('Performance Distribution by Task\n(Pass@1 for Code Gen, F1 for Others)', fontsize=14)
ax.grid(axis='y', alpha=0.3)

# Add mean markers
means = [d.mean() for d in task_data]
ax.scatter(range(1, 4), means, marker='D', color='black', s=50, zorder=5, label='Mean')
ax.legend()

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'task_performance_boxplot.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'task_performance_boxplot.png'}")

## 4. Design Comparison (SA vs DA vs MA)

In [ ]:
# Performance by design for each task
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, task in enumerate(tasks_order):
    ax = axes[idx]
    task_df = perf_df[perf_df['task'] == task]
    
    # Group by design
    design_means = task_df.groupby('design')['primary_metric_pct'].mean()
    design_stds = task_df.groupby('design')['primary_metric_pct'].std()
    
    designs = ['SA', 'DA', 'MA']
    x = range(len(designs))
    
    bars = ax.bar(x, [design_means.get(d, 0) for d in designs],
                  yerr=[design_stds.get(d, 0) for d in designs],
                  color=[DESIGN_COLORS[d] for d in designs],
                  edgecolor='black', linewidth=1, capsize=5, alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(designs)
    ax.set_ylabel(METRIC_LABELS[task])
    ax.set_title(TASK_LABELS[task])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, [design_means.get(d, 0) for d in designs]):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

fig.suptitle('Performance by Agent Design (SA vs DA vs MA)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'design_comparison_by_task.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'design_comparison_by_task.png'}")

In [ ]:
# Design comparison table
design_task_summary = perf_df.groupby(['task', 'design']).agg({
    'primary_metric_pct': 'mean',
    'accuracy_pct': 'mean'
}).round(2).unstack(level=0)

print("="*80)
print("MEAN PERFORMANCE BY DESIGN AND TASK")
print("="*80)
print(design_task_summary)

## 5. Prompting Strategy Comparison (Zero-shot vs Few-shot)

In [ ]:
# Performance by prompting strategy for each task
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, task in enumerate(tasks_order):
    ax = axes[idx]
    task_df = perf_df[perf_df['task'] == task]
    
    # Group by prompting and design
    grouped = task_df.groupby(['design', 'prompting'])['primary_metric_pct'].mean().unstack()
    
    x = np.arange(len(grouped.index))
    width = 0.35
    
    if 'zero-shot' in grouped.columns:
        bars1 = ax.bar(x - width/2, grouped['zero-shot'], width, 
                       label='Zero-shot', color=PROMPTING_COLORS['zero-shot'],
                       edgecolor='black', linewidth=1)
    if 'few-shot' in grouped.columns:
        bars2 = ax.bar(x + width/2, grouped['few-shot'], width,
                       label='Few-shot', color=PROMPTING_COLORS['few-shot'],
                       edgecolor='black', linewidth=1)
    
    ax.set_xticks(x)
    ax.set_xticklabels(grouped.index)
    ax.set_ylabel(METRIC_LABELS[task])
    ax.set_title(TASK_LABELS[task])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Zero-shot vs Few-shot Performance by Design', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'prompting_comparison_by_task.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'prompting_comparison_by_task.png'}")

In [ ]:
# Prompting comparison table
prompting_summary = perf_df.groupby(['task', 'prompting']).agg({
    'primary_metric_pct': 'mean'
}).round(2).unstack(level=1)

prompting_summary['delta'] = prompting_summary[('primary_metric_pct', 'few-shot')] - prompting_summary[('primary_metric_pct', 'zero-shot')]

print("="*80)
print("PROMPTING STRATEGY COMPARISON")
print("="*80)
print(prompting_summary)
print("\nPositive delta = Few-shot outperforms Zero-shot")

## 6. Mode Comparison (Instruct vs Thinking)

In [ ]:
# Performance by mode for each task
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

mode_colors = {'instruct': '#3498db', 'thinking': '#e74c3c'}

for idx, task in enumerate(tasks_order):
    ax = axes[idx]
    task_df = perf_df[perf_df['task'] == task]
    
    # Group by mode and design
    grouped = task_df.groupby(['design', 'mode'])['primary_metric_pct'].mean().unstack()
    
    x = np.arange(len(grouped.index))
    width = 0.35
    
    if 'instruct' in grouped.columns:
        bars1 = ax.bar(x - width/2, grouped['instruct'], width,
                       label='Instruct', color=mode_colors['instruct'],
                       edgecolor='black', linewidth=1)
    if 'thinking' in grouped.columns:
        bars2 = ax.bar(x + width/2, grouped['thinking'], width,
                       label='Thinking', color=mode_colors['thinking'],
                       edgecolor='black', linewidth=1, hatch='//')
    
    ax.set_xticks(x)
    ax.set_xticklabels(grouped.index)
    ax.set_ylabel(METRIC_LABELS[task])
    ax.set_title(TASK_LABELS[task])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Instruct vs Thinking Mode Performance by Design', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'mode_comparison_by_task.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'mode_comparison_by_task.png'}")

In [ ]:
# Mode comparison table
mode_summary = perf_df.groupby(['task', 'mode']).agg({
    'primary_metric_pct': 'mean'
}).round(2).unstack(level=1)

mode_summary['delta'] = mode_summary[('primary_metric_pct', 'thinking')] - mode_summary[('primary_metric_pct', 'instruct')]

print("="*80)
print("MODE COMPARISON (INSTRUCT vs THINKING)")
print("="*80)
print(mode_summary)
print("\nPositive delta = Thinking outperforms Instruct")

## 7. Model Size Scaling Analysis

In [ ]:
# Performance by model size for each task
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_sizes = [4, 8, 30, 49]

for idx, task in enumerate(tasks_order):
    ax = axes[idx]
    task_df = perf_df[perf_df['task'] == task]
    
    # Group by model size
    size_means = task_df.groupby('parameters_b')['primary_metric_pct'].mean()
    size_stds = task_df.groupby('parameters_b')['primary_metric_pct'].std()
    
    x = range(len(model_sizes))
    bars = ax.bar(x, [size_means.get(s, 0) for s in model_sizes],
                  yerr=[size_stds.get(s, 0) for s in model_sizes],
                  color=[MODEL_COLORS[s] for s in model_sizes],
                  edgecolor='black', linewidth=1, capsize=5, alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s}B' for s in model_sizes])
    ax.set_xlabel('Model Size')
    ax.set_ylabel(METRIC_LABELS[task])
    ax.set_title(TASK_LABELS[task])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, [size_means.get(s, 0) for s in model_sizes]):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

fig.suptitle('Performance by Model Size (4B → 8B → 30B → 49B)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'model_size_scaling.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'model_size_scaling.png'}")

In [ ]:
# Model size scaling line plot
fig, ax = plt.subplots(figsize=(10, 6))

for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    size_means = task_df.groupby('parameters_b')['primary_metric_pct'].mean()
    
    ax.plot(model_sizes, [size_means.get(s, np.nan) for s in model_sizes],
            marker='o', markersize=10, linewidth=2,
            color=TASK_COLORS[task], label=TASK_LABELS[task])

ax.set_xlabel('Model Size (Billion Parameters)', fontsize=12)
ax.set_ylabel('Performance (%)', fontsize=12)
ax.set_title('Model Size Scaling: Performance Trends', fontsize=14)
ax.set_xticks(model_sizes)
ax.set_xticklabels([f'{s}B' for s in model_sizes])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'model_size_scaling_lines.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'model_size_scaling_lines.png'}")

## 8. Heatmap: Full Experiment Matrix

In [ ]:
# Create heatmap for each task
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for idx, task in enumerate(tasks_order):
    ax = axes[idx]
    task_df = perf_df[perf_df['task'] == task].copy()
    
    # Create pivot table: rows = model config, cols = design
    task_df['config'] = task_df.apply(
        lambda r: f"{int(r['parameters_b'])}B {r['mode'][:4]} {r['prompting'][:4]}", axis=1)
    
    pivot = task_df.pivot_table(
        values='primary_metric_pct',
        index='config',
        columns='design',
        aggfunc='mean'
    )
    
    # Sort by model size
    pivot = pivot.reindex(sorted(pivot.index, key=lambda x: int(x.split('B')[0])))
    
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', 
                center=50, ax=ax, cbar_kws={'label': METRIC_LABELS[task]},
                linewidths=0.5, linecolor='white')
    
    ax.set_title(TASK_LABELS[task], fontsize=12)
    ax.set_xlabel('Design')
    ax.set_ylabel('Model Configuration')

fig.suptitle('Performance Heatmap: Model Config × Design', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'performance_heatmap.png', bbox_inches='tight')
plt.show()
print(f"Saved: {ANALYSIS_DIR / 'performance_heatmap.png'}")

## 9. Best Configuration Analysis

In [ ]:
print("="*80)
print("BEST CONFIGURATIONS BY TASK")
print("="*80)

for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    best_idx = task_df['primary_metric_pct'].idxmax()
    best = task_df.loc[best_idx]
    
    print(f"\n{TASK_LABELS[task]}:")
    print(f"  Best: {best['model']} | {best['design']} | {best['mode']} | {best['prompting']}")
    print(f"  Performance: {best['primary_metric_pct']:.2f}%")
    
    # Also show worst
    worst_idx = task_df['primary_metric_pct'].idxmin()
    worst = task_df.loc[worst_idx]
    print(f"  Worst: {worst['model']} | {worst['design']} | {worst['mode']} | {worst['prompting']}")
    print(f"  Performance: {worst['primary_metric_pct']:.2f}%")
    print(f"  Range: {best['primary_metric_pct'] - worst['primary_metric_pct']:.2f}%")

In [ ]:
# Top 5 configurations per task
print("="*80)
print("TOP 5 CONFIGURATIONS PER TASK")
print("="*80)

for task in tasks_order:
    print(f"\n{TASK_LABELS[task]}:")
    task_df = perf_df[perf_df['task'] == task].nlargest(5, 'primary_metric_pct')
    for i, (_, row) in enumerate(task_df.iterrows(), 1):
        print(f"  {i}. {row['short_label']} | {row['design']} | {row['primary_metric_pct']:.2f}%")

## 10. Summary Statistics

In [ ]:
# Comprehensive summary table
summary_table = perf_df.groupby(['task', 'design', 'mode', 'prompting']).agg({
    'primary_metric_pct': 'mean',
    'parameters_b': 'first'  # Just to show we have data
}).round(2)

print("="*80)
print("FULL EXPERIMENT SUMMARY")
print("="*80)
print(summary_table.to_string())

In [ ]:
# Save all outputs
perf_df.to_csv(ANALYSIS_DIR / 'comprehensive_performance_data.csv', index=False)
summary_table.to_csv(ANALYSIS_DIR / 'comprehensive_summary.csv')

print(f"\nSaved data to: {ANALYSIS_DIR}")
print("\nGenerated files:")
for f in sorted(ANALYSIS_DIR.glob('*')):
    print(f"  - {f.name}")

## 11. Key Findings Summary

In [ ]:
print("="*80)
print("KEY FINDINGS")
print("="*80)

# 1. Task difficulty
print("\n1. TASK DIFFICULTY RANKING:")
task_means = perf_df.groupby('task')['primary_metric_pct'].mean().sort_values(ascending=False)
for task, mean in task_means.items():
    print(f"   {TASK_LABELS[task]}: {mean:.2f}%")

# 2. Best design per task
print("\n2. OPTIMAL DESIGN PER TASK:")
for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    best_design = task_df.groupby('design')['primary_metric_pct'].mean().idxmax()
    best_val = task_df.groupby('design')['primary_metric_pct'].mean().max()
    print(f"   {TASK_LABELS[task]}: {best_design} ({best_val:.2f}%)")

# 3. Prompting effect
print("\n3. PROMPTING EFFECT (Few-shot vs Zero-shot):")
for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    few = task_df[task_df['prompting'] == 'few-shot']['primary_metric_pct'].mean()
    zero = task_df[task_df['prompting'] == 'zero-shot']['primary_metric_pct'].mean()
    delta = few - zero
    print(f"   {TASK_LABELS[task]}: {delta:+.2f}% ({'Few-shot better' if delta > 0 else 'Zero-shot better'})")

# 4. Mode effect
print("\n4. MODE EFFECT (Thinking vs Instruct):")
for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    think = task_df[task_df['mode'] == 'thinking']['primary_metric_pct'].mean()
    inst = task_df[task_df['mode'] == 'instruct']['primary_metric_pct'].mean()
    delta = think - inst
    print(f"   {TASK_LABELS[task]}: {delta:+.2f}% ({'Thinking better' if delta > 0 else 'Instruct better'})")

# 5. Model size effect
print("\n5. MODEL SIZE SCALING (4B vs 49B):")
for task in tasks_order:
    task_df = perf_df[perf_df['task'] == task]
    small = task_df[task_df['parameters_b'] == 4]['primary_metric_pct'].mean()
    large = task_df[task_df['parameters_b'] == 49]['primary_metric_pct'].mean()
    delta = large - small
    print(f"   {TASK_LABELS[task]}: {delta:+.2f}% ({'Larger better' if delta > 0 else 'Smaller better'})")